# Maintenance applicative et refactoring — TP Séance 4

Mes réponses aux 6 exercices.

## Exercice 1 – QCM sur les concepts

**1.1 Types de maintenance**

Associez chaque description au type de maintenance correspondant : correctif, évolutif, adaptatif, perfectif.

| Description | Type de maintenance |
| --- | --- |
| A. Corriger un bug qui empêche le programme de s'exécuter correctement | ? |
| B. Ajouter une nouvelle fonctionnalité demandée par l'utilisateur | ? |
| C. Adapter le code à un nouvel environnement technique (API, version Python) | ? |
| D. Refactoriser le code pour améliorer lisibilité et structure | ? |

**1.2 Tests et qualité**

1. Quel est le rôle principal d'un test unitaire ? a) style / b) fonction isolée / c) interface
2. Pourquoi utiliser des logs plutôt que `print` en production ? a) consultables plus tard / b) pour l'utilisateur / c) déco
3. Quel outil corrige automatiquement le style Python ? a) flake8 / b) black / c) SonarQube

**1.3 Refactoring — Vrai ou Faux**

1. Le refactoring doit toujours changer le comportement observable du code.
2. L'objectif du refactoring est de réduire la dette technique.
3. Isoler la logique métier dans des fonctions permet de faciliter les tests.

### Ma réponse

**1.1**
- A → **correctif** : on répare un bug qui existe déjà.
- B → **évolutif** : on ajoute une nouvelle fonctionnalité demandée.
- C → **adaptatif** : c'est l'environnement (l'API, la version de Python) qui change, le code doit s'adapter.
- D → **perfectif** : on améliore la qualité du code sans changer ce qu'il fait.

**1.2**
1. **b)** — un test unitaire teste une fonction toute seule, isolée du reste.
2. **a)** — en prod on n'a pas l'écran de l'utilisateur, les logs restent et on peut les relire plus tard. Les `print` sont perdus.
3. **b)** — black reformate le code automatiquement. flake8 ne fait que signaler les erreurs, il ne corrige pas.

**1.3**
1. **Faux** — c'est le contraire : le refactoring change la structure interne **sans** changer le comportement observable.
2. **Vrai** — le but c'est justement de réduire la dette technique.
3. **Vrai** — une fonction isolée se teste facilement, sans avoir à lancer toute l'appli.

## Exercice 2 – Identifier la dette technique

```python
tasks = []

def add_task(t):
    if t != "":
        tasks.append([t, False])
    else:
        print("vide")

def mark_done(index):
    tasks[index][1] = True
```

**Questions :**
1. Quels problèmes de lisibilité et de structure observez-vous ?
2. Quel type de dette technique représente l'usage de `tasks.append([t, False])` ?
3. Proposez au moins deux améliorations pour rendre le code testable et maintenable.

### Ma réponse

**1. Problèmes que je vois :**
- `tasks` est une **variable globale**, donc les fonctions modifient un état partagé. C'est dur à tester parce qu'un test laisse des données pour le suivant.
- Le nom `t` ne veut rien dire, on ne sait pas si c'est le texte de la tâche ou autre chose.
- La tâche est stockée comme `[t, False]` : une liste où `[0]` = le texte et `[1]` = si c'est fait. Le souci c'est qu'il faut connaître l'ordre par cœur. Quand on lit `tasks[index][1] = True`, on ne comprend pas tout de suite que `[1]` veut dire "fait".
- `print("vide")` au lieu de renvoyer quelque chose : la fonction ne retourne rien, donc on ne peut pas savoir si l'ajout a marché ou pas.
- `mark_done` n'est pas protégée : si `index` n'existe pas, ça plante (IndexError).

**2. Type de dette technique :**
C'est de la dette **structurelle** (mauvaise façon de modéliser les données). Stocker une tâche dans une liste `[texte, fait]` au lieu d'un objet avec des noms, ça crée des "index magiques" : `[1]` ne dit pas ce qu'il représente. Et si demain on veut ajouter un champ (une date par ex.), tout décale et on risque de casser le code partout.

**3. Mes améliorations :**

- Utiliser un **dictionnaire** au lieu d'une liste, pour que chaque champ ait un nom :

```python
tasks.append({"task": task, "done": False})
# et pour marquer fait :
tasks[index]["done"] = True   # beaucoup plus clair que [1]
```

- Enlever la variable globale et **passer `tasks` en paramètre**, et **renvoyer un booléen** au lieu de faire un print :

```python
def add_task(tasks, task):
    if task.strip() == "":
        return False
    tasks.append({"task": task, "done": False})
    return True
```

Comme ça la fonction est testable : on lui donne une liste, et on vérifie ce qu'elle renvoie.

## Exercice 3 – Type de maintenance à appliquer

Pour chaque situation, indiquez le type de maintenance et justifiez :
1. Un utilisateur signale qu'un programme plante lorsqu'il saisit 0 comme montant.
2. Le client demande l'ajout d'un bouton pour inverser les devises dans le convertisseur.
3. L'API de taux de change change de version et l'application ne récupère plus les taux.
4. Le code contient des listes imbriquées et tout est écrit dans le flux principal, lecture difficile.

### Ma réponse

1. **Correctif** — le programme plante avec une entrée valide (0), c'est un bug à réparer.
2. **Évolutif** — on ajoute une nouvelle fonctionnalité (le bouton d'inversion) demandée par le client.
3. **Adaptatif** — le code n'a pas de bug, c'est l'API extérieure qui a changé de version. Le code doit s'adapter pour continuer à marcher. (Attention, c'est le piège : ce n'est pas correctif.)
4. **Perfectif** — le code marche mais il est mal écrit. On le refactore pour qu'il soit plus lisible, sans changer ce qu'il fait.

## Exercice 4 – Refactoring

```python
def convert(amount, from_currency, to_currency):
    rates = {"EUR": 1, "USD": 1.1}
    if from_currency == "EUR" and to_currency == "USD":
        return amount * 1.1
    if from_currency == "USD" and to_currency == "EUR":
        return amount / 1.1
    return amount
```

**Questions :**
1. Identifiez au moins deux problèmes (style, duplication, extensibilité).
2. Refactorez la fonction pour : utiliser un dictionnaire de taux, supporter toutes les conversions sans écrire chaque condition, être testable.

### Ma réponse

**1. Les problèmes :**
- Le taux `1.1` est écrit plusieurs fois en dur. Si le taux change, il faut le modifier à plusieurs endroits → risque d'oubli.
- Le dictionnaire `rates` est créé mais jamais utilisé, c'est du code mort. Toute la logique passe par les `if`.
- Ça ne s'étend pas : pour ajouter une 3e devise il faudrait écrire un `if` pour chaque paire possible. Ça devient vite ingérable.

**2. Mon refactoring :**

L'idée c'est de prendre une devise de référence (EUR, taux = 1). Chaque taux dit combien vaut 1 EUR dans la devise. Du coup une seule formule suffit pour toutes les conversions :
- on convertit d'abord le montant en EUR : `amount / rates[from]`
- puis de l'EUR vers la devise cible : `* rates[to]`

(le code est dans la cellule en dessous)

In [ ]:
def convert(amount, from_currency, to_currency):
    rates = {"EUR": 1, "USD": 1.1, "GBP": 0.85}  # pour ajouter une devise = juste une ligne

    if amount < 0:
        return None
    if from_currency not in rates or to_currency not in rates:
        return None

    amount_in_eur = amount / rates[from_currency]   # on passe par l'EUR
    return amount_in_eur * rates[to_currency]


# petits essais
print(convert(10, "EUR", "USD"))  # 11.0
print(convert(11, "USD", "EUR"))  # 10.0
print(convert(50, "EUR", "EUR"))  # 50.0
print(convert(-5, "EUR", "USD"))  # None

## Exercice 5 – Tests unitaires

Écrivez un test unitaire pour la fonction refactorisée de l'exercice précédent.
1. Vérifiez que 10 EUR → USD donne le bon résultat.
2. Vérifiez que la conversion d'une devise vers elle-même retourne le même montant.
3. Vérifiez qu'un montant négatif retourne `None`.

### Ma réponse

J'écris les tests avec des `assert` : si l'assertion est fausse ça lève une erreur, sinon tout passe.
La fonction `convert` est définie dans la cellule de l'exercice 4, donc il faut l'avoir exécutée avant.

(Dans un vrai projet on mettrait ça dans un fichier `test_convertisseur.py` et on lancerait `uv run pytest`, mais ici dans le notebook je teste direct.)

In [ ]:
def test_eur_to_usd():
    # 10 EUR -> USD = 10 * 1.1 = 11.0
    assert convert(10, "EUR", "USD") == 11.0

def test_same_currency():
    # une devise vers elle-meme = meme montant
    assert convert(50, "EUR", "EUR") == 50

def test_negative_amount():
    # montant negatif -> None
    assert convert(-5, "EUR", "USD") is None

test_eur_to_usd()
test_same_currency()
test_negative_amount()
print("Tous les tests passent ✅")

## Exercice 6 – Analyse de code et justification

```python
tasks = []

def add_task(task):
    if task.strip() == "":
        return False
    tasks.append({"task": task, "done": False})
    return True
```

**Questions :**
1. Ce code est-il testable ? Pourquoi ?
2. Quelle modification pour permettre le test unitaire de plusieurs tâches indépendantes ?
3. Classez les modifications (corrective / évolutive / adaptative / perfective) si on devait ajouter :
   - un contrôle que la tâche n'existe pas déjà
   - l'enregistrement des tâches dans `st.session_state` pour Streamlit
   - un changement du dictionnaire en liste imbriquée

### Ma réponse

**1. Est-ce testable ?**
Pas vraiment, à cause de la variable globale `tasks`. Comme la liste est partagée par tous les appels, les tests ne sont pas indépendants : un test ajoute une tâche, et elle reste dans la liste pour le test suivant. Du coup les résultats dépendent de l'ordre des tests, ce qui n'est pas fiable.
(Le bon point quand même : la fonction renvoie `True`/`False`, donc on peut vérifier le retour.)

**2. La modification à faire :**
Enlever la globale et passer `tasks` en paramètre. Comme ça chaque test crée sa propre liste vide au début :

```python
def add_task(tasks, task):
    if task.strip() == "":
        return False
    tasks.append({"task": task, "done": False})
    return True

# dans les tests :
def test_add():
    tasks = []                       # liste neuve, juste pour ce test
    assert add_task(tasks, "A") is True
    assert len(tasks) == 1
```

Chaque test a son état à lui, ils ne se polluent plus entre eux.

**3. Classement des modifications :**
- Ajouter un contrôle anti-doublon → **évolutif** (nouvelle règle / fonctionnalité qui n'existait pas).
- Enregistrer dans `st.session_state` pour Streamlit → **adaptatif** (on s'adapte au fonctionnement du framework Streamlit).
- Repasser le dictionnaire en liste imbriquée → **perfectif** sur le papier (changement de structure interne), mais en vrai c'est une mauvaise idée : on revient aux index magiques, donc on rajoute de la dette au lieu d'en enlever.